In [ ]:
%%writefile _baseline_nb_tests.py
# Stack integration tests for the jupyter/baseline notebook image (RHAIENG-6389).
# Emitted by test_notebook.ipynb via %%writefile for subprocess unittest isolation.
# Keep this suite aligned with the lean phase-1 dependency set in pyproject.toml
# (JupyterLab + Elyra/Kale/PDF; no datascience / DB-connector stack).

from pathlib import Path
import json
import re
import unittest
from platform import python_version
import importlib.metadata
import subprocess
import sys
import tempfile


def get_major_minor(s):
    return '.'.join(s.split('.')[:2])


def load_expected_versions() -> dict:
    lock_file = Path('./expected_versions.json')
    if not lock_file.exists():
        # Phase 1 baseline has no imagestream yet; version asserts soft-skip.
        return {}
    with open(lock_file, 'r') as file:
        return json.load(file)


def get_expected_version(dependency_name: str) -> str | None:
    raw_value = expected_versions.get(dependency_name)
    if raw_value is None:
        return None
    raw_version = re.sub(r'^\D+', '', str(raw_value))
    return get_major_minor(raw_version)


class TestPythonVersion(unittest.TestCase):
    def test_version(self):
        expected_major_minor = get_expected_version('Python')
        if expected_major_minor is None:
            self.skipTest('expected_versions.json missing Python')
        actual_major_minor = get_major_minor(python_version())
        self.assertEqual(actual_major_minor, expected_major_minor, 'incorrect version')


class TestJupyterLab(unittest.TestCase):
    def test_version(self):
        import jupyterlab
        expected_major_minor = get_expected_version('JupyterLab')
        if expected_major_minor is None:
            self.skipTest('expected_versions.json missing JupyterLab')
        actual_major_minor = get_major_minor(jupyterlab.__version__)
        self.assertEqual(actual_major_minor, expected_major_minor, 'incorrect version')

    def test_import(self):
        import jupyterlab
        self.assertTrue(jupyterlab.__version__)


class TestOdhElyra(unittest.TestCase):
    def test_import_and_version(self):
        import elyra
        ver = getattr(elyra, '__version__', None) or importlib.metadata.version('odh-elyra')
        self.assertTrue(ver)
        expected_major_minor = get_expected_version('Odh-Elyra')
        if expected_major_minor is None:
            return
        self.assertEqual(get_major_minor(ver), expected_major_minor, 'incorrect version')


class TestKfp(unittest.TestCase):
    def test_import(self):
        import kfp
        self.assertTrue(hasattr(kfp, 'dsl'))


class TestKale(unittest.TestCase):
    def test_import(self):
        import kale
        self.assertTrue(importlib.metadata.version('odh-kale'))


class TestPandoc(unittest.TestCase):
    def test_pandoc_cli(self):
        r = subprocess.run(
            ['pandoc', '--version'],
            capture_output=True,
            text=True,
            timeout=60,
        )
        self.assertEqual(r.returncode, 0, r.stderr or r.stdout)
        self.assertIn('pandoc', (r.stdout or '').lower())


class TestBuildTools(unittest.TestCase):
    def test_setuptools_and_wheel_import(self):
        import setuptools
        import wheel
        self.assertTrue(len(setuptools.__version__) > 0)
        self.assertTrue(len(wheel.__version__) > 0)


class TestUvAndMicropipenv(unittest.TestCase):
    def test_uv_cli_version(self):
        r = subprocess.run(
            [sys.executable, '-m', 'uv', '--version'],
            capture_output=True,
            text=True,
            timeout=60,
        )
        self.assertEqual(r.returncode, 0, r.stderr or r.stdout)
        self.assertIn('uv', (r.stdout or '').lower())

    def test_micropipenv_import(self):
        import micropipenv
        self.assertTrue(hasattr(micropipenv, 'install'))


class TestPipWorkflow(unittest.TestCase):
    def test_pip_list_invokable(self):
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'list', '--format=columns'],
            capture_output=True,
            text=True,
            timeout=120,
        )
        self.assertEqual(r.returncode, 0, r.stderr or r.stdout)
        self.assertIn('jupyterlab', r.stdout.lower())

    def test_pip_show_jupyterlab(self):
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'show', 'jupyterlab'],
            capture_output=True,
            text=True,
            timeout=60,
        )
        self.assertEqual(r.returncode, 0, r.stderr)
        self.assertIn('Name: jupyterlab', r.stdout)

    def test_pip_install_noop_dry_run_to_tmp(self):
        with tempfile.TemporaryDirectory() as tmp:
            r = subprocess.run(
                [
                    sys.executable,
                    '-m',
                    'pip',
                    'install',
                    '--dry-run',
                    '--no-deps',
                    '--target',
                    tmp,
                    'jupyterlab',
                ],
                capture_output=True,
                text=True,
                timeout=180,
            )
            self.assertEqual(r.returncode, 0, r.stdout + r.stderr)


STACK_TEST_CLASS_NAMES: tuple[str, ...] = tuple(
    name
    for name, obj in sorted(globals().items())
    if isinstance(obj, type)
    and issubclass(obj, unittest.TestCase)
    and obj is not unittest.TestCase
    and name.startswith('Test')
)

expected_versions = load_expected_versions()



In [ ]:
# Run each TestCase in a subprocess (RHAIENG-4983: peak RSS / OOM on ppc64le).
import os
import subprocess
import sys
from pathlib import Path

_root = Path.cwd().resolve()
sys.path.insert(0, str(_root))

from _baseline_nb_tests import STACK_TEST_CLASS_NAMES

failures = []
for cls_name in STACK_TEST_CLASS_NAMES:
    rc = subprocess.run(
        [
            sys.executable,
            "-m",
            "unittest",
            "-v",
            f"_baseline_nb_tests.{cls_name}",
        ],
        cwd=str(_root),
        env=os.environ.copy(),
    ).returncode
    if rc != 0:
        failures.append(cls_name)

if failures:
    print("FAILED: " + ", ".join(failures), file=sys.stderr)
    sys.exit(1)

